# 🚑 XGBoost Rescue — Optuna + Clinical Weights
**Problem:** Clarke A+B = 92.91% (FAIL, target >95%). Zone D = 4.8%.  
**Fix:** Clinical sample weights + Optuna hyperparameter tuning.

---

In [ ]:
!pip install polars xgboost optuna plotly pyarrow scikit-learn -q

import polars as pl
import numpy as np
import xgboost as xgb
import optuna
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import mean_absolute_error, mean_squared_error
from datetime import datetime, timedelta
import os, gc, time, json

from google.colab import drive
drive.mount('/content/drive')

print(f"XGBoost: {xgb.__version__}")
print(f"Optuna : {optuna.__version__}")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════
BASE_PATH   = "/content/drive/MyDrive/DiabetesKDS/data"
INPUT_FILE  = f"{BASE_PATH}/processed/train_elite_physio_engineered.parquet"
MODEL_FILE  = f"{BASE_PATH}/models/xgboost_optuna_v2.json"

FEATURE_COLS = [
    "CGM", "IOB", "COB", "Vel_5min", "Vel_15min", "Acc_15min",
    "Roll_Mean_60", "Roll_Std_60", "Hour_Sin", "Hour_Cos",
    "Insulin_Sensitivity_Risk",
]
TARGET_COL  = "Target_CGM"
TRAIN_RATIO = 0.80
N_TRIALS    = 50  # Optuna trials (increase for better results)

os.makedirs(os.path.dirname(MODEL_FILE), exist_ok=True)
print(f"📁 Input : {INPUT_FILE} — exists: {os.path.exists(INPUT_FILE)}")
print(f"📁 Model : {MODEL_FILE}")

---
## STEP 1 — Load & Split (Time-Series, No Leakage)

In [ ]:
df = pl.read_parquet(INPUT_FILE)
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]
print(f"Loaded: {df.shape[0]:,} rows, using {len(FEATURE_COLS)} features")

df = df.sort("datetime").drop_nulls(subset=FEATURE_COLS + [TARGET_COL])
split_idx = int(len(df) * TRAIN_RATIO)

df_train = df.head(split_idx)
df_test  = df.tail(len(df) - split_idx)

X_train = df_train.select(FEATURE_COLS).to_numpy().astype(np.float32)
y_train = df_train[TARGET_COL].to_numpy().astype(np.float32)
X_test  = df_test.select(FEATURE_COLS).to_numpy().astype(np.float32)
y_test  = df_test[TARGET_COL].to_numpy().astype(np.float32)

test_ids = df_test["id"].to_list()
test_datetimes = df_test["datetime"].to_list()

del df, df_train; gc.collect()
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

---
## STEP 2 — Clinical Sample Weights 🏥
Force the model to prioritize dangerous glucose ranges.

In [ ]:
def compute_clinical_weights(y):
    """Assign higher weights to dangerous glucose zones."""
    w = np.ones_like(y, dtype=np.float32)
    w[y < 70]                        = 10.0   # Hypo: CRITICAL
    w[y > 250]                       = 5.0    # Severe Hyper
    w[(y >= 180) & (y <= 250)]       = 2.0    # Mild Hyper
    # Normal [70, 180] stays at 1.0
    return w

w_train = compute_clinical_weights(y_train)
w_test  = compute_clinical_weights(y_test)

print("╔═══════════════════════════════════════════════╗")
print("║      🏥  CLINICAL WEIGHT DISTRIBUTION        ║")
print("╠═══════════════════════════════════════════════╣")
for label, cond in [("Hypo (<70)", w_train==10), ("Hyper (>250)", w_train==5),
                     ("Mild Hyper", w_train==2), ("Normal", w_train==1)]:
    n = cond.sum()
    print(f"║  {label:18s}: {n:>10,} ({n/len(w_train)*100:>5.2f}%)  ║")
print(f"║  Effective dataset: {w_train.sum():>10,.0f} weighted   ║")
print("╚═══════════════════════════════════════════════╝")

---
## STEP 3 — Optuna Hyperparameter Search 🔬

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Clarke Zone classifier (vectorized, fast)
# ══════════════════════════════════════════════════════════════════════
def clarke_zone_ab_pct(ref, pred):
    """Fast Clarke A+B percentage (vectorized)."""
    ref, pred = np.asarray(ref, float), np.asarray(pred, float)
    n = len(ref)
    # Zone A: within 20% or both < 70
    zone_a = ((np.abs(pred - ref) <= 0.2 * ref) | ((ref <= 70) & (pred <= 70)))
    # Zone B: same side of 70/180 thresholds but outside 20%
    zone_b = (~zone_a) & (
        ((ref >= 180) & (pred >= 180)) |
        ((ref <= 70) & (pred <= 70)) |
        ((ref >= 70) & (ref <= 180) & (pred >= 70) & (pred <= 180)) |
        ((pred >= 0.8 * ref) & (pred <= 1.2 * ref))
    )
    return (zone_a.sum() + zone_b.sum()) / n * 100, zone_a.sum() / n * 100


# ══════════════════════════════════════════════════════════════════════
# Optuna Objective
# ══════════════════════════════════════════════════════════════════════
def objective(trial):
    params = {
        "objective":        "reg:squarederror",
        "tree_method":      "hist",
        "device":           "cuda",
        "eval_metric":      "rmse",
        "n_estimators":     3000,
        "early_stopping_rounds": 50,
        "verbosity":        0,
        "random_state":     42,
        # ── Search Space ──────────────────────────
        "learning_rate":    trial.suggest_float("learning_rate", 0.001, 0.1, log=True),
        "max_depth":        trial.suggest_int("max_depth", 4, 12),
        "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":        trial.suggest_float("reg_alpha", 0.0, 10.0),
        "reg_lambda":       trial.suggest_float("reg_lambda", 0.0, 10.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "gamma":            trial.suggest_float("gamma", 0.0, 5.0),
    }

    model = xgb.XGBRegressor(**params)
    model.fit(
        X_train, y_train,
        sample_weight=w_train,
        eval_set=[(X_test, y_test)],
        sample_weight_eval_set=[w_test],
        verbose=False,
    )

    y_pred = np.clip(model.predict(X_test), 20, 500)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    # Store Clarke A+B as a user attribute for analysis
    ab_pct, _ = clarke_zone_ab_pct(y_test, y_pred)
    trial.set_user_attr("clarke_ab", ab_pct)
    trial.set_user_attr("best_iteration", model.best_iteration)

    del model; gc.collect()
    return rmse


print("✅ Objective function defined.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# RUN OPTUNA
# ══════════════════════════════════════════════════════════════════════
t0 = time.time()
optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(
    direction="minimize",
    study_name="glucose_rescue_v2",
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=100),
)

# Progress callback
def progress_callback(study, trial):
    if (trial.number + 1) % 5 == 0 or trial.number == 0:
        best = study.best_trial
        ab = best.user_attrs.get('clarke_ab', 0)
        print(f"  Trial {trial.number+1:3d}/{N_TRIALS} | "
              f"Best RMSE: {best.value:.4f} | Clarke A+B: {ab:.2f}% | "
              f"Elapsed: {time.time()-t0:.0f}s")

print(f"🔬 Starting Optuna with {N_TRIALS} trials...")
print("=" * 65)
study.optimize(objective, n_trials=N_TRIALS, callbacks=[progress_callback])

elapsed = time.time() - t0
print(f"\n✅ Optuna complete in {elapsed:.0f}s ({elapsed/60:.1f} min)")
print(f"   Best RMSE: {study.best_value:.4f}")
print(f"   Clarke A+B: {study.best_trial.user_attrs['clarke_ab']:.2f}%")

In [ ]:
# ── Best Hyperparameters ─────────────────────────────────────────────
best_params = study.best_trial.params

print("╔══════════════════════════════════════════════════╗")
print("║        🏆  BEST HYPERPARAMETERS                 ║")
print("╠══════════════════════════════════════════════════╣")
for k, v in best_params.items():
    if isinstance(v, float):
        print(f"║  {k:25s}: {v:>12.6f}       ║")
    else:
        print(f"║  {k:25s}: {v:>12}       ║")
print("╚══════════════════════════════════════════════════╝")

# Optuna optimization history
trials_df = study.trials_dataframe()
fig = px.line(
    trials_df, x="number", y="value",
    title="📉 Optuna Trial History (RMSE)",
    labels={"number": "Trial", "value": "RMSE"},
)
fig.update_layout(template="plotly_dark", height=350)
fig.show()

---
## STEP 4 — Train Final Model with Best Params 🚀

In [ ]:
final_params = {
    "objective": "reg:squarederror",
    "tree_method": "hist", "device": "cuda",
    "n_estimators": 3000, "early_stopping_rounds": 50,
    "eval_metric": "rmse", "random_state": 42, "verbosity": 1,
    **best_params,
}

print("🚀 Training final model with best params...")
t_train = time.time()

final_model = xgb.XGBRegressor(**final_params)
final_model.fit(
    X_train, y_train,
    sample_weight=w_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    sample_weight_eval_set=[w_train, w_test],
    verbose=200,
)

train_time = time.time() - t_train
print(f"\n✅ Done in {train_time:.1f}s | Best iter: {final_model.best_iteration}")

In [ ]:
# ── Learning Curves ──────────────────────────────────────────────────
res = final_model.evals_result()
fig = go.Figure()
fig.add_trace(go.Scatter(y=res["validation_0"]["rmse"], name="Train", line=dict(color="#4ECDC4")))
fig.add_trace(go.Scatter(y=res["validation_1"]["rmse"], name="Test", line=dict(color="#FF6B6B")))
fig.add_vline(x=final_model.best_iteration, line_dash="dash", line_color="white",
              annotation_text=f"Best: {final_model.best_iteration}")
fig.update_layout(title="📉 Learning Curves (Final Model)", template="plotly_dark",
                  height=400, xaxis_title="Round", yaxis_title="RMSE")
fig.show()

---
## STEP 5 — Evaluation & Clarke Error Grid 🩸

In [ ]:
y_pred = np.clip(final_model.predict(X_test), 20, 500)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
mask = y_test > 0
mape = np.mean(np.abs((y_test[mask] - y_pred[mask]) / y_test[mask])) * 100
errors = np.abs(y_test - y_pred)
within_10 = np.mean(errors <= 10) * 100
within_15 = np.mean(errors <= 15) * 100
within_20 = np.mean(errors <= 20) * 100

print("╔══════════════════════════════════════════════════════╗")
print("║         📊  v2 TEST METRICS                        ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  RMSE       : {rmse:>10.4f} mg/dL                  ║")
print(f"║  MAE        : {mae:>10.4f} mg/dL                  ║")
print(f"║  MAPE       : {mape:>10.4f}%                      ║")
print(f"║  Within ±10 : {within_10:>10.2f}%                    ║")
print(f"║  Within ±15 : {within_15:>10.2f}%                    ║")
print(f"║  Within ±20 : {within_20:>10.2f}%                    ║")
print("╚══════════════════════════════════════════════════════╝")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Clarke Error Grid (Full Implementation)
# ══════════════════════════════════════════════════════════════════════
def clarke_zones_full(ref, pred):
    ref, pred = np.asarray(ref, float), np.asarray(pred, float)
    zones = np.full(len(ref), 'D', dtype='U1')
    for i in range(len(ref)):
        r, p = ref[i], pred[i]
        if (r <= 70 and p <= 70) or abs(p - r) <= 0.2 * r:
            zones[i] = 'A'
        elif r <= 70 and p >= 180:
            zones[i] = 'E'
        elif r >= 180 and p <= 70:
            zones[i] = 'E'
        elif 70 <= r <= 180 and (p < 70 or p > 180):
            zones[i] = 'C'
        elif (r < 70 and 70 < p <= 180) or (r > 180 and 70 <= p < 180):
            zones[i] = 'D'
        else:
            zones[i] = 'B'
    return zones

zones = clarke_zones_full(y_test, y_pred)
unique, counts = np.unique(zones, return_counts=True)
zs = dict(zip(unique, counts))
total = len(zones)
zone_ab = (zs.get('A',0) + zs.get('B',0)) / total * 100

print("╔══════════════════════════════════════════════════╗")
print("║       🩸  CLARKE ERROR GRID RESULTS (v2)        ║")
print("╠══════════════════════════════════════════════════╣")
for z in ['A','B','C','D','E']:
    n = zs.get(z, 0); pct = n/total*100
    bar = '█' * int(pct/2)
    print(f"║  Zone {z}: {n:>10,} ({pct:>6.2f}%)  {bar:20s}  ║")
print(f"║                                                  ║")
status = "✅ PASS" if zone_ab > 95 else "⚠️  BELOW"
print(f"║  A+B = {zone_ab:.2f}%  (Target >95%)  {status:12s}   ║")
print(f"║  v1 was 92.91%  →  Δ = {zone_ab - 92.91:+.2f}%                 ║")
print("╚══════════════════════════════════════════════════╝")

In [ ]:
# ── Clarke Error Grid Plot ───────────────────────────────────────────
sample_n = min(100000, len(y_test))
idx = np.random.RandomState(42).choice(len(y_test), sample_n, replace=False)
z_s = clarke_zones_full(y_test[idx], y_pred[idx])
colors = {'A':'#2ECC71','B':'#F39C12','C':'#E74C3C','D':'#9B59B6','E':'#C0392B'}

fig = go.Figure()
fig.add_trace(go.Scattergl(
    x=y_test[idx], y=y_pred[idx], mode='markers',
    marker=dict(size=2, color=[colors.get(z,'gray') for z in z_s], opacity=0.4),
    hoverinfo='skip',
))
fig.add_trace(go.Scatter(x=[0,400], y=[0,400], mode='lines',
    line=dict(color='white', width=1, dash='dash'), name='Perfect'))

# 20% boundaries
x_l = np.linspace(1, 400, 300)
fig.add_trace(go.Scatter(x=x_l, y=x_l*1.2, mode='lines',
    line=dict(color='rgba(46,204,113,0.4)', width=1), showlegend=False))
fig.add_trace(go.Scatter(x=x_l, y=x_l*0.8, mode='lines',
    line=dict(color='rgba(46,204,113,0.4)', width=1), showlegend=False))

for v in [70, 180]:
    fig.add_hline(y=v, line_dash='dot', line_color='rgba(255,255,255,0.2)')
    fig.add_vline(x=v, line_dash='dot', line_color='rgba(255,255,255,0.2)')

fig.add_annotation(x=340, y=60, text=f"Zone A: {zs.get('A',0)/total*100:.1f}%",
    font=dict(size=14, color='#2ECC71'), showarrow=False)
fig.add_annotation(x=340, y=40, text=f"A+B: {zone_ab:.1f}%",
    font=dict(size=16, color='white'), showarrow=False)

fig.update_layout(
    title="🩸 Clarke Error Grid — Optuna v2 (Clinical Weights)",
    xaxis_title="Reference (mg/dL)", yaxis_title="Predicted (mg/dL)",
    template="plotly_dark", height=650, width=650,
    xaxis=dict(range=[0,400]), yaxis=dict(range=[0,400]),
)
fig.show()

---
## STEP 6 — Feature Importance & Persistence Bias Check

In [ ]:
importance = final_model.get_booster().get_score(importance_type="gain")
feat_imp = {}
for key, val in importance.items():
    i = int(key.replace("f","")); feat_imp[FEATURE_COLS[i]] = val

feat_imp = dict(sorted(feat_imp.items(), key=lambda x: x[1], reverse=True))
total_gain = sum(feat_imp.values())

print("Feature Importance (Gain):")
print("=" * 55)
for name, val in feat_imp.items():
    pct = val / total_gain * 100
    bar = '█' * int(pct)
    flag = " ⚠️ DOMINANT" if pct > 50 else ""
    print(f"  {name:30s} {pct:>6.2f}%  {bar}{flag}")

cgm_pct = feat_imp.get('CGM', 0) / total_gain * 100
print(f"\n🔍 CGM share: {cgm_pct:.1f}% {'(Persistence bias reduced ✅)' if cgm_pct < 60 else '(Still dominant ⚠️)'}")

fig = px.bar(
    x=list(feat_imp.values()), y=list(feat_imp.keys()), orientation='h',
    title='🏆 Feature Importance — v2 (Optuna + Clinical Weights)',
    color=list(feat_imp.values()), color_continuous_scale='Viridis',
)
fig.update_layout(template='plotly_dark', height=400, yaxis=dict(autorange='reversed'))
fig.show()

In [ ]:
# ── 24h Prediction Trace ─────────────────────────────────────────────
test_id_arr = np.array(test_ids)
uid, ucounts = np.unique(test_id_arr, return_counts=True)
best_pid = uid[np.argmax(ucounts)]
pm = test_id_arr == best_pid
p_dt = np.array(test_datetimes)[pm]

mid = len(p_dt)//2
t0_w, t1_w = p_dt[mid], p_dt[mid] + timedelta(hours=24)
wm = (p_dt >= t0_w) & (p_dt <= t1_w)

fig = go.Figure()
fig.add_trace(go.Scatter(x=p_dt[wm], y=y_test[pm][wm], name='Actual',
    line=dict(color='#00D4AA', width=2)))
fig.add_trace(go.Scatter(x=p_dt[wm], y=y_pred[pm][wm], name='Predicted (+30m)',
    line=dict(color='#FF6B6B', width=2, dash='dot')))
fig.add_hline(y=70, line_dash='dash', line_color='red', annotation_text='Hypo')
fig.add_hline(y=180, line_dash='dash', line_color='orange', annotation_text='Hyper')
fig.update_layout(title=f"📉 24h Trace — Patient '{best_pid}'",
    template='plotly_dark', height=450, xaxis_title='Time', yaxis_title='mg/dL')
fig.show()

---
## STEP 7 — Save Model & Final Verdict 💾

In [ ]:
final_model.save_model(MODEL_FILE)
model_size = os.path.getsize(MODEL_FILE) / (1024**2)

meta = {
    "model": "xgboost_optuna_v2", "created": datetime.now().isoformat(),
    "features": FEATURE_COLS, "target": TARGET_COL, "horizon_min": 30,
    "best_iteration": int(final_model.best_iteration),
    "optuna_trials": N_TRIALS,
    "best_params": {k: round(v,6) if isinstance(v,float) else v for k,v in best_params.items()},
    "metrics": {
        "rmse": round(float(rmse),4), "mae": round(float(mae),4),
        "mape": round(float(mape),4),
        "within_10": round(float(within_10),2),
        "within_15": round(float(within_15),2),
        "within_20": round(float(within_20),2),
        "clarke_a_pct": round(zs.get('A',0)/total*100, 2),
        "clarke_ab_pct": round(float(zone_ab), 2),
    },
    "clinical_weights": {"hypo_lt70": 10.0, "hyper_gt250": 5.0, "mild_hyper": 2.0, "normal": 1.0},
}

with open(MODEL_FILE.replace('.json','_metadata.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print(f"💾 Model: {MODEL_FILE} ({model_size:.1f} MB)")
print(f"📋 Metadata saved.")

In [ ]:
print("")
print("╔══════════════════════════════════════════════════════════════════╗")
print("║          🏁  MODEL RESCUE REPORT  (v1 → v2)                   ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║                          v1 (old)     v2 (new)     Change     ║")
print("║  ─────────────────────────────────────────────────────────     ║")
print(f"║  Clarke A+B:           92.91%      {zone_ab:>6.2f}%     {zone_ab-92.91:>+6.2f}%    ║")
print(f"║  RMSE:                  —          {rmse:>8.4f}                  ║")
print(f"║  MAE:                   —          {mae:>8.4f}                  ║")
print(f"║  Zone D:               ~4.80%      {zs.get('D',0)/total*100:>6.2f}%     {zs.get('D',0)/total*100-4.80:>+6.2f}%    ║")
print("║                                                                ║")
print(f"║  CGM Importance:         —         {cgm_pct:>6.1f}%                  ║")
print(f"║  Optuna Trials:                    {N_TRIALS:>6}                    ║")
print(f"║  Best Iteration:                   {final_model.best_iteration:>6}                    ║")
print("║                                                                ║")

verdict = "✅ RESCUED — Target Achieved!" if zone_ab > 95 else "⚠️  IMPROVED — More trials may help"
print(f"║  VERDICT: {verdict:53s} ║")
print("╚══════════════════════════════════════════════════════════════════╝")

gc.collect()
print("\n🧹 Done.")